# EDA – Data Understanding
**Customer Health & Sentiment Scoring Multi-Agent AI Solution**

| | |
|---|---|
| **Stack** | Python + DuckDB |
| **Input** | `data/bronze/{customers,transactions,engagements,interactions}.parquet` (ZSTD) |
| **Output** | `reports/eda/` – one CSV per check, PNG charts, `eda_summary.md` |
| **Period / grain** | 1 Oct 2025 – 30 Sep 2026, `customer_id × month` |

**How to use:** run the cells top to bottom (*Run All*). Steps 5–11 need the customer-month table built in Section 4.
Each step shows its key tables inline; full results are saved as CSVs. Lines marked **[FLAG]** are auto-detected findings.

## 0. Install packages (run once, then comment out)

In [ ]:
# %pip install pandas duckdb matplotlib pyarrow

## 1. Imports and configuration (change settings here only)

In [ ]:
import time
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

%matplotlib inline
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:,.3f}".format)

BRONZE_DIR = Path("data/bronze")          # or full path, e.g. Path(r"C:\path\to\project\data\bronze")
OUT_DIR = Path("reports/eda")
WORK_DB = OUT_DIR / "eda_work.duckdb"     # caches the customer-month table between sessions

WINDOW_START = "2025-10-01"               # inclusive
WINDOW_END = "2026-10-01"                 # exclusive (data ends 30 Sep 2026)

MIN_HISTORY_MONTHS = 3        # months observed (incl. current) before a customer-month is scored
CHURN_GAP_MONTHS = 3          # inactive >= N months since last activity -> churn proxy
RARE_CAT_PCT = 1.0            # category share below this % = rare
MOM_ALERT_PCT = 30.0          # month-on-month change above this % = flag
SEGMENT_SPREAD_ALERT = 1.30   # max/min segment mean above this = segment-relative scoring needed
HIGH_CORR = 0.80              # |spearman| above this = redundant features
TEXT_SAMPLE_PER_LABEL = 34    # ~100 rows for manual reading
CORR_SAMPLE_ROWS = 500_000
MIN_COHORT_CUSTOMERS = 20_000 # sample-size gate for a candidate cohort
SEED = 42
DUCKDB_THREADS = 4            # reduce to 2 on a low-RAM laptop
DUCKDB_MEMORY = "4GB"         # reduce to "2GB" if the kernel crashes
REBUILD_CM = False            # True = force rebuild of the customer-month table
SHOW_ROWS = 10                # rows displayed per table in the notebook

SPEND_TYPES = ("purchase", "subscription_payment", "bill_payment")   # gross spend = completed only
COMPLAINT_TYPES = ("complaint", "billing_dispute", "cancellation_request")
OPEN_RESOLUTION = ("unresolved", "escalated", "pending")

TABLES = {  # table -> primary key
    "customers": "customer_id",
    "transactions": "transaction_id",
    "engagements": "engagement_id",
    "interactions": "interaction_id",
}

# Cohort definitions (SQL on table cm)
HISTORY_COND = f"active AND months_observed >= {MIN_HISTORY_MONTHS}"
CORE_COND = f"{HISTORY_COND} AND has_tx AND (has_eng OR has_int)"

## 2. Connection and helper functions

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
try:                       # safe to re-run this cell
    con.close()
except NameError:
    pass
con = duckdb.connect(str(WORK_DB))
con.execute(f"SET threads={DUCKDB_THREADS}")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}'")
con.execute(f"SET temp_directory='{(OUT_DIR / 'duckdb_tmp').as_posix()}'")   # spill to disk instead of crashing
con.execute("SET preserve_insertion_order=false")
FINDINGS = []


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


def sql_list(values) -> str:
    return "(" + ", ".join(f"'{v}'" for v in values) + ")"


def yes(col: str) -> str:
    return f"(upper(trim(CAST({col} AS VARCHAR))) IN ('Y','YES','TRUE','1'))"


def header(title: str):
    display(Markdown(f"### {title}"))


def save(df: pd.DataFrame, name: str, show: int = SHOW_ROWS) -> pd.DataFrame:
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"--- {name}  ({len(df):,} rows) -> {path}")
    if show:
        display(df.head(show))
    return df


def flag(step: str, msg: str):
    FINDINGS.append((step, msg))
    print(f"  [FLAG] {msg}")


def columns_of(view: str) -> list:
    return q(f"DESCRIBE {view}")["column_name"].tolist()


def table_exists(name: str) -> bool:
    return bool(q(f"SELECT count(*) n FROM information_schema.tables WHERE table_name = '{name}'").iloc[0, 0])


def run_step(fn):
    t0 = time.time()
    fn()
    print(f"\n(done in {time.time() - t0:,.1f}s)")

print("DuckDB", duckdb.__version__, "| pandas", pd.__version__, "| output ->", OUT_DIR.resolve())

## 3. Register sources (raw views + typed views)

In [ ]:
def register_sources():
    for t in TABLES:
        file_ = BRONZE_DIR / f"{t}.parquet"
        folder = BRONZE_DIR / t
        if file_.exists():
            src = f"read_parquet('{file_.as_posix()}')"
        elif folder.exists():
            src = f"read_parquet('{(folder / '**' / '*.parquet').as_posix()}')"
        else:
            raise FileNotFoundError(f"Parquet for '{t}' not found in {BRONZE_DIR.resolve()}")
        con.execute(f"CREATE OR REPLACE VIEW raw_{t} AS SELECT * FROM {src}")

    win = f"TIMESTAMP '{WINDOW_START}'", f"TIMESTAMP '{WINDOW_END}'"

    con.execute("""
        CREATE OR REPLACE VIEW cust AS
        SELECT *, TRY_CAST(signup_date AS DATE) AS signup,
               TRY_CAST(age AS DOUBLE) AS age_n, TRY_CAST(tenure_months AS DOUBLE) AS tenure_n
        FROM raw_customers
    """)
    con.execute("""
        CREATE OR REPLACE VIEW cust1 AS            -- one row per customer (guards against duplicates)
        SELECT * FROM cust
        QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY signup NULLS LAST) = 1
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW tx AS
        SELECT *, CAST(date_trunc('month', ts) AS DATE) AS mth,
               (ts >= {win[0]} AND ts < {win[1]}) AS in_window
        FROM (SELECT *, TRY_CAST(transaction_ts AS TIMESTAMP) AS ts,
                     TRY_CAST(amount AS DOUBLE) AS amt, TRY_CAST(discount_amount AS DOUBLE) AS disc
              FROM raw_transactions)
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW eng AS
        SELECT *, CAST(date_trunc('month', ts) AS DATE) AS mth,
               (ts >= {win[0]} AND ts < {win[1]}) AS in_window
        FROM (SELECT *, TRY_CAST(event_ts AS TIMESTAMP) AS ts,
                     TRY_CAST(session_duration_sec AS DOUBLE) AS dur,
                     TRY_CAST(pages_viewed AS DOUBLE) AS pages,
                     {yes('is_converted')} AS converted
              FROM raw_engagements)
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW itx AS
        SELECT *, CAST(date_trunc('month', ts) AS DATE) AS mth,
               (ts >= {win[0]} AND ts < {win[1]}) AS in_window
        FROM (SELECT *, TRY_CAST(interaction_ts AS TIMESTAMP) AS ts,
                     TRY_CAST(duration_sec AS DOUBLE) AS dur,
                     TRY_CAST(csat_score AS DOUBLE) AS csat,
                     TRY_CAST(nps_score AS DOUBLE) AS nps,
                     TRY_CAST(sentiment_score_gt AS DOUBLE) AS sscore,
                     CAST(interaction_text AS VARCHAR) AS txt,
                     {yes('first_contact_resolution')} AS fcr
              FROM raw_interactions)
    """)


EVENT_VIEWS = {"transactions": "tx", "engagements": "eng", "interactions": "itx"}
TS_COL = {"transactions": "transaction_ts", "engagements": "event_ts", "interactions": "interaction_ts"}

register_sources()
display(pd.DataFrame([{"table": t, "rows": q(f"SELECT count(*) FROM raw_{t}").iloc[0, 0]} for t in TABLES]))

## 4. Build the customer-month table `cm` (used by Steps 5–11)
One row per customer × month (customers × 12 months). First build takes a few minutes; afterwards it is reused from `eda_work.duckdb`.

In [ ]:
def build_cm(force: bool = False):
    if table_exists("cm") and not force:
        print("cm table already exists in work DB (set REBUILD_CM = True to refresh)")
        return
    t0 = time.time()
    print("Building customer-month table cm ...")
    con.execute(f"""
    CREATE OR REPLACE TABLE cm AS
    WITH months AS (
        SELECT CAST(range AS DATE) AS mth
        FROM range(TIMESTAMP '{WINDOW_START}', TIMESTAMP '{WINDOW_END}', INTERVAL 1 MONTH)
    ),
    spine AS (
        SELECT c.customer_id, m.mth,
               (c.signup IS NOT NULL AND m.mth < date_trunc('month', c.signup)) AS pre_signup
        FROM cust1 c CROSS JOIN months m
    ),
    tx_m AS (
        SELECT customer_id, mth,
            count(*)                                                    AS n_tx,
            count_if(transaction_status = 'completed')                  AS n_tx_completed,
            count_if(transaction_status = 'failed')                     AS n_tx_failed,
            count_if(transaction_status = 'reversed')                   AS n_tx_reversed,
            count_if(transaction_type = 'refund')                       AS n_refunds,
            coalesce(sum(amt) FILTER (WHERE transaction_status = 'completed'
                     AND transaction_type IN {sql_list(SPEND_TYPES)}), 0) AS gross_spend,
            coalesce(sum(abs(amt)) FILTER (WHERE transaction_status = 'completed'
                     AND transaction_type = 'refund'), 0)               AS refund_amt,
            coalesce(sum(disc), 0)                                      AS discount_amt,
            mode(currency)                                              AS tx_currency
        FROM tx WHERE in_window GROUP BY ALL
    ),
    eng_m AS (
        SELECT customer_id, mth,
            count(*)                                  AS n_eng,
            count_if(event_type = 'login')            AS n_logins,
            count_if(event_type = 'offer_redeem')     AS n_redeem,
            count_if(converted)                       AS n_conversions,
            coalesce(sum(dur), 0)                     AS eng_seconds,
            coalesce(sum(pages), 0)                   AS pages_viewed,
            count(DISTINCT engagement_channel)        AS n_eng_channels
        FROM eng WHERE in_window GROUP BY ALL
    ),
    itx_m AS (
        SELECT customer_id, mth,
            count(*)                                                       AS n_int,
            count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)})      AS n_complaint,
            count_if(interaction_type = 'cancellation_request')           AS n_cancel_req,
            count_if(sentiment_label = 'negative')                        AS n_negative,
            count_if(sentiment_label = 'positive')                        AS n_positive,
            count_if(resolution_status IN {sql_list(OPEN_RESOLUTION)})     AS n_open_issue,
            count_if(fcr)                                                  AS n_fcr,
            avg(sscore)                                                    AS avg_sentiment_gt,
            avg(csat) AS avg_csat, count(csat) AS n_csat,
            avg(nps)  AS avg_nps,  count(nps)  AS n_nps
        FROM itx WHERE in_window GROUP BY ALL
    ),
    base AS (
        SELECT s.customer_id, s.mth, s.pre_signup,
               (t.customer_id IS NOT NULL) AS has_tx,
               (e.customer_id IS NOT NULL) AS has_eng,
               (i.customer_id IS NOT NULL) AS has_int,
               t.* EXCLUDE (customer_id, mth),
               e.* EXCLUDE (customer_id, mth),
               i.* EXCLUDE (customer_id, mth)
        FROM spine s
        LEFT JOIN tx_m  t ON s.customer_id = t.customer_id AND s.mth = t.mth
        LEFT JOIN eng_m e ON s.customer_id = e.customer_id AND s.mth = e.mth
        LEFT JOIN itx_m i ON s.customer_id = i.customer_id AND s.mth = i.mth
    ),
    lc AS (
        SELECT *,
            CAST(has_tx AS INT) + CAST(has_eng AS INT) + CAST(has_int AS INT) AS n_sources,
            (has_tx OR has_eng OR has_int) AS active
        FROM base
    ),
    lc2 AS (
        SELECT *,
            min(CASE WHEN active THEN mth END) OVER (PARTITION BY customer_id)           AS first_active_mth,
            max(CASE WHEN active THEN mth END) OVER w_prev                               AS last_active_before,
            sum(CAST(active AS INT)) OVER w_cum                                          AS active_months_to_date,
            lag(active) OVER w_ord                                                       AS active_prev
        FROM lc
        WINDOW w_ord  AS (PARTITION BY customer_id ORDER BY mth),
               w_cum  AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
               w_prev AS (PARTITION BY customer_id ORDER BY mth ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
    )
    SELECT *,
        CASE WHEN first_active_mth IS NOT NULL AND mth >= first_active_mth
             THEN date_diff('month', first_active_mth, mth) + 1 ELSE 0 END             AS months_observed,
        date_diff('month', last_active_before, mth)                                    AS months_since_last_active,
        CASE WHEN active AND mth = first_active_mth          THEN 'new_in_data'
             WHEN active AND active_prev                     THEN 'retained'
             WHEN active                                     THEN 'reactivated'
             WHEN last_active_before IS NULL                 THEN 'not_yet_active'
             WHEN date_diff('month', last_active_before, mth) < {CHURN_GAP_MONTHS} THEN 'dormant'
             ELSE 'churn_proxy' END                                                    AS lifecycle_status,
        coalesce(nullif(concat_ws('+', CASE WHEN has_tx THEN 'TX' END,
                                       CASE WHEN has_eng THEN 'ENG' END,
                                       CASE WHEN has_int THEN 'INT' END), ''), 'none')  AS source_combo,
        CASE WHEN has_tx THEN percent_rank() OVER (PARTITION BY mth, tx_currency ORDER BY gross_spend) END
                                                                                       AS spend_pctile
    FROM lc2
    """)
    n = q("SELECT count(*) FROM cm").iloc[0, 0]
    print(f"cm built: {n:,} rows in {time.time() - t0:,.1f}s")


build_cm(force=REBUILD_CM)
display(q("SELECT * FROM cm WHERE active LIMIT 5"))

## Step 1 – Structure, grain, duplicates, join keys

In [ ]:
def step1_structure():
    header("STEP 1 - Structure, grain, duplicates, join keys")
    rows = []
    for t, pk in TABLES.items():
        schema = q(f"DESCRIBE raw_{t}")[["column_name", "column_type"]]
        save(schema.assign(table=t), f"s1_schema_{t}", show=0)
        r = q(f"""SELECT count(*) n_rows, count(DISTINCT {pk}) n_distinct_pk,
                         count(DISTINCT customer_id) n_customers FROM raw_{t}""").iloc[0]
        full_dup = q(f"SELECT (SELECT count(*) FROM raw_{t}) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_{t}))").iloc[0, 0]
        rows.append({"table": t, "n_rows": r.n_rows, "n_cols": len(schema), "primary_key": pk,
                     "pk_duplicates": r.n_rows - r.n_distinct_pk, "full_row_duplicates": full_dup,
                     "n_customers": r.n_customers,
                     "rows_per_customer": round(r.n_rows / max(r.n_customers, 1), 2),
                     "grain": "1 row = 1 customer" if t == "customers" else f"1 row = 1 {t[:-1]} event"})
        if r.n_rows - r.n_distinct_pk > 0:
            flag("S1", f"{t}: {r.n_rows - r.n_distinct_pk:,} duplicate {pk} values")
        if full_dup > 0:
            flag("S1", f"{t}: {full_dup:,} fully duplicated rows")
    save(pd.DataFrame(rows), "s1_table_overview")

    rows = []
    for t, v in EVENT_VIEWS.items():
        r = q(f"""
            SELECT min(ts) min_ts, max(ts) max_ts,
                   count_if(ts IS NULL) unparseable_ts,
                   count_if(NOT in_window) outside_window,
                   count_if(left(CAST(month AS VARCHAR), 7) <> strftime(ts, '%Y-%m')) month_col_mismatch,
                   count(DISTINCT mth) n_months
            FROM {v}""").iloc[0].to_dict()
        rows.append({"table": t, **r})
        if r["outside_window"] > 0:
            flag("S1", f"{t}: {r['outside_window']:,} rows outside {WINDOW_START}..{WINDOW_END}")
        if r["month_col_mismatch"] > 0:
            flag("S1", f"{t}: {r['month_col_mismatch']:,} rows where 'month' != month of {TS_COL[t]}")
    save(pd.DataFrame(rows), "s1_date_ranges")

    rows = []
    for t in EVENT_VIEWS:
        r = q(f"""
            SELECT count(DISTINCT e.customer_id) n_customers,
                   count(DISTINCT e.customer_id) FILTER (WHERE c.customer_id IS NULL) orphan_customers,
                   count_if(c.customer_id IS NULL) orphan_rows,
                   count_if(e.customer_id IS NULL) null_customer_id_rows
            FROM raw_{t} e LEFT JOIN (SELECT DISTINCT customer_id FROM raw_customers) c
              ON e.customer_id = c.customer_id""").iloc[0].to_dict()
        rows.append({"table": t, **r, "pct_customers_found": round(100 * (1 - r["orphan_customers"] / max(r["n_customers"], 1)), 2)})
        if r["orphan_customers"] > 0:
            flag("S1", f"{t}: {r['orphan_customers']:,} customer_ids not in customers.csv")
    save(pd.DataFrame(rows), "s1_orphan_ids")

    r = q("""
        SELECT count(*) n_customers,
               count_if(customer_id NOT IN (SELECT customer_id FROM raw_transactions)
                    AND customer_id NOT IN (SELECT customer_id FROM raw_engagements)
                    AND customer_id NOT IN (SELECT customer_id FROM raw_interactions)) no_activity
        FROM (SELECT DISTINCT customer_id FROM raw_customers)""").iloc[0]
    pct = 100 * r.no_activity / r.n_customers
    save(pd.DataFrame([{"n_customers": r.n_customers, "customers_without_any_activity": r.no_activity,
                        "pct": round(pct, 2)}]), "s1_customers_without_activity")
    if pct > 10:
        flag("S1", f"{pct:.1f}% of customers have no activity in any source in the period")


run_step(step1_structure)

## Step 2 – Data quality: nulls, invalid values, statuses, outliers

In [ ]:
def step2_quality():
    header("STEP 2 - Nulls, invalid values, statuses, outliers")

    # 2a. Null / blank % for every column of every table
    frames = []
    for t in TABLES:
        cs = columns_of(f"raw_{t}")
        expr = ", ".join(
            f"""round(100.0 * count_if("{c}" IS NULL OR trim(CAST("{c}" AS VARCHAR)) IN
                ('', 'NA', 'N/A', 'null', 'NULL', 'None', 'nan')) / count(*), 3) AS "{c}" """ for c in cs)
        d = q(f"SELECT {expr} FROM raw_{t}").T.reset_index()
        d.columns = ["column", "null_or_blank_pct"]
        frames.append(d.assign(table=t))
    nulls = save(pd.concat(frames)[["table", "column", "null_or_blank_pct"]]
                 .sort_values("null_or_blank_pct", ascending=False), "s2_null_profile", show=15)
    for _, r in nulls[nulls.null_or_blank_pct > 5].iterrows():
        flag("S2", f"{r.table}.{r.column}: {r.null_or_blank_pct:.1f}% null/blank - check if expected")

    # 2b. Do nulls cluster in certain months?
    save(q("""
        SELECT mth, count(*) n,
               round(100.0*count_if(csat IS NULL)/count(*), 2) csat_null_pct,
               round(100.0*count_if(nps IS NULL)/count(*), 2)  nps_null_pct,
               round(100.0*count_if(txt IS NULL OR trim(txt) = '')/count(*), 2) text_blank_pct,
               round(100.0*count_if(sscore IS NULL)/count(*), 2) sentiment_gt_null_pct
        FROM itx GROUP BY 1 ORDER BY 1"""), "s2_nulls_by_month_interactions", show=12)
    save(q("""
        SELECT mth, count(*) n,
               round(100.0*count_if(campaign_id IS NULL OR trim(CAST(campaign_id AS VARCHAR))='')/count(*), 2) campaign_null_pct,
               round(100.0*count_if(app_version IS NULL OR trim(CAST(app_version AS VARCHAR))='')/count(*), 2) app_version_null_pct
        FROM eng GROUP BY 1 ORDER BY 1"""), "s2_nulls_by_month_engagements", show=0)

    # 2c. Invalid-value rules
    rules = [
        ("transactions", "tx", "amount is null", "amt IS NULL"),
        ("transactions", "tx", "amount < 0", "amt < 0"),
        ("transactions", "tx", "amount = 0", "amt = 0"),
        ("transactions", "tx", "discount < 0", "disc < 0"),
        ("transactions", "tx", "discount > amount", "disc > abs(amt)"),
        ("transactions", "tx", "future timestamp", "ts > now()"),
        ("engagements", "eng", "session_duration < 0", "dur < 0"),
        ("engagements", "eng", "session_duration > 4h", "dur > 14400"),
        ("engagements", "eng", "pages_viewed < 0", "pages < 0"),
        ("interactions", "itx", "duration < 0", "dur < 0"),
        ("interactions", "itx", "duration > 4h", "dur > 14400"),
        ("interactions", "itx", "csat outside 1-5 (assumed scale)", "csat NOT BETWEEN 1 AND 5"),
        ("interactions", "itx", "nps outside 0-10", "nps NOT BETWEEN 0 AND 10"),
        ("interactions", "itx", "sentiment_score_gt outside [-1,1]", "sscore NOT BETWEEN -1 AND 1"),
        ("interactions", "itx", "label positive but score < 0", "sentiment_label='positive' AND sscore < 0"),
        ("interactions", "itx", "label negative but score > 0", "sentiment_label='negative' AND sscore > 0"),
        ("interactions", "itx", "text blank", "txt IS NULL OR trim(txt) = ''"),
        ("customers", "cust", "age < 18 or > 100", "age_n < 18 OR age_n > 100"),
        ("customers", "cust", "tenure_months < 0", "tenure_n < 0"),
        ("customers", "cust", "signup after data window", f"signup >= DATE '{WINDOW_END}'"),
        ("customers", "cust", "signup_date unparseable/null", "signup IS NULL"),
    ]
    rows = []
    for t, v, name, cond in rules:
        n_bad, n = (int(x) for x in q(f"SELECT count_if({cond}) n_bad, count(*) n FROM {v}").iloc[0])
        rows.append({"table": t, "rule": name, "n_violations": n_bad, "pct": round(100 * n_bad / n, 3)})
        if n_bad > 0:
            flag("S2", f"{t}: '{name}' -> {n_bad:,} rows ({100 * n_bad / n:.2f}%)")
    save(pd.DataFrame(rows), "s2_invalid_value_rules", show=25)

    # 2d. Transaction status x type (defines gross vs net spend)
    save(q("""
        SELECT transaction_type, transaction_status, count(*) n,
               round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_rows,
               count_if(amt < 0) n_negative_amount, round(median(amt), 2) median_amount
        FROM tx GROUP BY ALL ORDER BY n DESC"""), "s2_tx_status_by_type", show=20)
    save(q("""
        SELECT currency, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct,
               count(DISTINCT customer_id) n_customers, round(median(amt), 2) median_amount
        FROM tx GROUP BY 1 ORDER BY n DESC"""), "s2_tx_currency_mix")
    n_cur = q("SELECT count(DISTINCT currency) FROM tx").iloc[0, 0]
    if n_cur > 1:
        flag("S2", f"{n_cur} currencies in transactions - do NOT sum amounts across currencies; "
                   "use FX conversion or within-currency percentiles")
    multi = q("""SELECT count(*) FROM (SELECT customer_id FROM tx GROUP BY 1
                 HAVING count(DISTINCT currency) > 1)""").iloc[0, 0]
    if multi > 0:
        flag("S2", f"{multi:,} customers transact in more than one currency")
    save(q("""SELECT resolution_status, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s2_resolution_status")

    # 2e. Outliers - per-customer volumes and amounts (percentiles, not mean)
    rows = []
    for t, v in EVENT_VIEWS.items():
        rows.append(q(f"""
            WITH c AS (SELECT customer_id, count(*) n FROM {v} WHERE in_window GROUP BY 1)
            SELECT '{t}' source, count(*) customers_with_activity, min(n) min,
                   quantile_cont(n, 0.5) p50, quantile_cont(n, 0.95) p95,
                   quantile_cont(n, 0.99) p99, max(n) max FROM c"""))
    save(pd.concat(rows), "s2_events_per_customer_percentiles")
    save(q("""WITH p AS (SELECT currency, quantile_cont(amt, 0.99) p99 FROM tx GROUP BY 1)
              SELECT t.currency, round(quantile_cont(t.amt, 0.01), 2) p01, round(median(t.amt), 2) p50,
                     round(quantile_cont(t.amt, 0.95), 2) p95, round(any_value(p.p99), 2) p99,
                     round(max(t.amt), 2) max, count_if(t.amt > 5 * p.p99) n_above_5x_p99
              FROM tx t JOIN p USING (currency) GROUP BY 1 ORDER BY 1"""), "s2_amount_outliers_by_currency")
    save(q("""SELECT customer_id, count(*) n_interactions, count_if(interaction_type = 'complaint') n_complaints
              FROM itx WHERE in_window GROUP BY 1 ORDER BY n_interactions DESC LIMIT 50"""),
         "s2_top_customers_by_interactions", show=5)


run_step(step2_quality)

## Step 3 – Univariate distributions

In [ ]:
NUMERIC = {"tx": ["amt", "disc"], "eng": ["dur", "pages"],
           "itx": ["dur", "csat", "nps", "sscore"], "cust": ["age_n", "tenure_n"]}
CATEGORICAL = {
    "tx": ["transaction_type", "product_category", "currency", "payment_method", "merchant_channel",
           "transaction_status", "is_international"],
    "eng": ["engagement_channel", "event_type", "device_type", "app_version", "is_converted"],
    "itx": ["channel", "interaction_type", "resolution_status", "first_contact_resolution", "sentiment_label"],
    "cust": ["gender", "region", "country", "customer_segment", "acquisition_channel", "income_band",
             "preferred_channel", "loyalty_tier", "marketing_consent"],
}


def step3_univariate():
    header("STEP 3 - Univariate distributions")
    frames = []
    for v, cs in NUMERIC.items():
        for c in cs:
            frames.append(q(f"""
                SELECT '{v}' tbl, '{c}' col, count({c}) n_nonnull,
                       round(100.0*count_if({c} = 0)/count(*), 2) zero_pct,
                       round(100.0*count_if({c} < 0)/count(*), 2) neg_pct,
                       min({c}) min, quantile_cont({c}, 0.01) p01, quantile_cont({c}, 0.25) p25,
                       median({c}) p50, quantile_cont({c}, 0.75) p75, quantile_cont({c}, 0.95) p95,
                       quantile_cont({c}, 0.99) p99, max({c}) max, avg({c}) mean, stddev({c}) std,
                       skewness({c}) skew FROM {v}"""))
    num = save(pd.concat(frames), "s3_numeric_profile", show=20)
    for _, r in num.iterrows():
        if pd.notna(r["skew"]) and abs(r["skew"]) > 1:
            flag("S3", f"{r.tbl}.{r.col} skew={r['skew']:.2f} -> consider log1p / capping at P99")

    frames, summary = [], []
    for v, cs in CATEGORICAL.items():
        for c in cs:
            d = q(f"""SELECT CAST({c} AS VARCHAR) value, count(*) n,
                             round(100.0*count(*)/sum(count(*)) OVER (), 3) pct
                      FROM {v} GROUP BY 1 ORDER BY n DESC""")
            d.insert(0, "col", c)
            d.insert(0, "tbl", v)
            frames.append(d)
            summary.append({"tbl": v, "col": c, "n_levels": len(d),
                            "n_rare_levels": int((d.pct < RARE_CAT_PCT).sum()),
                            "top_level": d.value.iloc[0], "top_pct": d.pct.iloc[0]})
    save(pd.concat(frames), "s3_categorical_frequencies", show=0)
    cat = save(pd.DataFrame(summary), "s3_categorical_summary", show=30)
    for _, r in cat[cat.n_rare_levels > 0].iterrows():
        flag("S3", f"{r.tbl}.{r.col}: {r.n_rare_levels} rare levels (<{RARE_CAT_PCT}%) -> group as 'Other'")


run_step(step3_univariate)

## Step 4 – Time patterns: monthly trends, seasonality, gaps, partial months

In [ ]:
def step4_time():
    header("STEP 4 - Monthly trends, seasonality, gaps, partial months")
    monthly = q(f"""
        WITH t AS (SELECT mth, count(*) tx_rows, count(DISTINCT customer_id) tx_customers,
                          round(100.0*count_if(transaction_status <> 'completed')/count(*), 2) tx_failed_or_reversed_pct,
                          round(sum(amt) FILTER (WHERE transaction_status='completed'
                                AND transaction_type IN {sql_list(SPEND_TYPES)} AND currency='USD'), 0) usd_gross_spend
                   FROM tx WHERE in_window GROUP BY 1),
             e AS (SELECT mth, count(*) eng_rows, count(DISTINCT customer_id) eng_customers
                   FROM eng WHERE in_window GROUP BY 1),
             i AS (SELECT mth, count(*) int_rows, count(DISTINCT customer_id) int_customers,
                          round(100.0*count_if(interaction_type IN {sql_list(COMPLAINT_TYPES)})/count(*), 2) complaint_pct,
                          round(100.0*count_if(sentiment_label='negative')/count(*), 2) negative_pct,
                          round(avg(csat), 3) avg_csat, round(avg(nps), 3) avg_nps
                   FROM itx WHERE in_window GROUP BY 1)
        SELECT * FROM t FULL JOIN e USING (mth) FULL JOIN i USING (mth) ORDER BY mth""")
    save(monthly, "s4_monthly_trends", show=12)

    vol_cols = ["tx_rows", "tx_customers", "usd_gross_spend", "eng_rows", "eng_customers", "int_rows", "int_customers"]
    mom = monthly[["mth"]].copy()
    for c in vol_cols:
        mom[c + "_mom_pct"] = (monthly[c].pct_change() * 100).round(2)
    save(mom, "s4_mom_change_pct", show=12)
    for c in vol_cols:
        for _, r in mom[mom[c + "_mom_pct"].abs() > MOM_ALERT_PCT].iterrows():
            flag("S4", f"{c}: {r[c + '_mom_pct']:+.1f}% MoM in {pd.Timestamp(r.mth):%Y-%m} -> real event or load issue?")

    frames = []
    for t, v in EVENT_VIEWS.items():
        frames.append(q(f"""
            SELECT '{t}' source, mth, count(DISTINCT CAST(ts AS DATE)) days_with_data,
                   day(last_day(mth)) calendar_days, min(ts) first_ts, max(ts) last_ts
            FROM {v} WHERE in_window GROUP BY 1, 2 ORDER BY 2"""))
    days = save(pd.concat(frames), "s4_days_with_data", show=0)
    for _, r in days[days.days_with_data < days.calendar_days].iterrows():
        flag("S4", f"{r.source} {pd.Timestamp(r.mth):%Y-%m}: data on {r.days_with_data}/{r.calendar_days} days (gap/partial month)")

    # Daily anomalies: day volume < 50% or > 200% of trailing 7-day median
    frames = []
    for t, v in EVENT_VIEWS.items():
        d = q(f"SELECT CAST(ts AS DATE) d, count(*) n FROM {v} WHERE in_window GROUP BY 1 ORDER BY 1")
        d["rolling_median_7d"] = d.n.rolling(7, min_periods=3).median().shift(1)
        d["ratio"] = (d.n / d.rolling_median_7d).round(2)
        frames.append(d[(d.ratio < 0.5) | (d.ratio > 2.0)].assign(source=t))
    anomalies = save(pd.concat(frames), "s4_daily_volume_anomalies", show=10)
    if len(anomalies):
        flag("S4", f"{len(anomalies)} days with abnormal volume (see s4_daily_volume_anomalies.csv)")

    frames = []
    for t, v in EVENT_VIEWS.items():
        frames.append(q(f"""SELECT '{t}' source, dayname(ts) day_of_week, isodow(ts) dow, count(*) n
                            FROM {v} WHERE in_window GROUP BY ALL ORDER BY dow"""))
    save(pd.concat(frames), "s4_day_of_week", show=0)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    x = pd.to_datetime(monthly.mth).dt.strftime("%Y-%m")
    for c in ["tx_rows", "eng_rows", "int_rows"]:
        ax[0].plot(x, monthly[c], marker="o", label=c)
    for c in ["tx_customers", "eng_customers", "int_customers"]:
        ax[1].plot(x, monthly[c], marker="o", label=c)
    ax[0].set_title("Monthly rows by source"); ax[1].set_title("Monthly active customers by source")
    for a in ax:
        a.tick_params(axis="x", rotation=45); a.legend(); a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "s4_monthly_trends.png", dpi=120); plt.show()


run_step(step4_time)

## Step 5 – Source coverage (drives confidence and cohort choice)

In [ ]:
def step5_coverage():
    header("STEP 5 - Coverage matrix across Transactions / Engagements / Interactions")
    save(q("""SELECT n_sources, count(*) n_customer_months,
                     round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_of_all_customer_months
              FROM cm WHERE NOT pre_signup GROUP BY 1 ORDER BY 1"""), "s5_n_sources_all_customer_months")
    combo = save(q("""SELECT source_combo, count(*) n_customer_months,
                             round(100.0*count(*)/sum(count(*)) OVER (), 2) pct_of_active
                      FROM cm WHERE active GROUP BY 1 ORDER BY n_customer_months DESC"""), "s5_source_combinations")
    by_month = save(q("""
        SELECT mth, count(*) all_customers, count_if(active) active_customers,
               round(100.0*count_if(active)/count(*), 2) pct_active,
               round(100.0*count_if(n_sources = 3)/nullif(count_if(active), 0), 2) pct_active_3_sources,
               round(100.0*count_if(n_sources = 2)/nullif(count_if(active), 0), 2) pct_active_2_sources,
               round(100.0*count_if(n_sources = 1)/nullif(count_if(active), 0), 2) pct_active_1_source
        FROM cm WHERE NOT pre_signup GROUP BY 1 ORDER BY 1"""), "s5_coverage_by_month", show=12)
    save(q("""SELECT months_with_2plus_sources, count(*) n_customers,
                     round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
              FROM (SELECT customer_id, count_if(n_sources >= 2) months_with_2plus_sources FROM cm GROUP BY 1)
              GROUP BY 1 ORDER BY 1"""), "s5_customers_by_months_with_2plus_sources", show=13)

    pct2 = q("SELECT 100.0*count_if(n_sources >= 2)/count(*) FROM cm WHERE active").iloc[0, 0]
    flag("S5", f"{pct2:.1f}% of active customer-months have >= 2 sources "
               f"({'good' if pct2 >= 50 else 'sparse -> confidence must reflect missing sources'})")
    flag("S5", f"On average {by_month.pct_active.mean():.1f}% of customers are active in a given month")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(combo.source_combo, combo.pct_of_active)
    ax.set_xlabel("% of active customer-months"); ax.set_title("Source combinations (active customer-months)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s5_source_combinations.png", dpi=120); plt.show()


run_step(step5_coverage)

## Step 6 – Customer lifecycle

In [ ]:
def step6_lifecycle():
    header("STEP 6 - Lifecycle: first/last seen, active months, status, MoM change")
    per_cust = save(q("""
        WITH c AS (SELECT customer_id, sum(CAST(active AS INT)) n_active_months FROM cm GROUP BY 1)
        SELECT n_active_months, count(*) n_customers, round(100.0*count(*)/sum(count(*)) OVER (), 2) pct
        FROM c GROUP BY 1 ORDER BY 1"""), "s6_active_months_distribution", show=13)
    save(q("""SELECT first_active_mth, count(DISTINCT customer_id) n_customers FROM cm
              WHERE first_active_mth IS NOT NULL GROUP BY 1 ORDER BY 1"""), "s6_first_active_month", show=0)
    save(q("""WITH c AS (SELECT customer_id, max(CASE WHEN active THEN mth END) last_active FROM cm GROUP BY 1)
              SELECT last_active, count(*) n_customers FROM c WHERE last_active IS NOT NULL GROUP BY 1 ORDER BY 1"""),
         "s6_last_active_month", show=0)
    flag("S6", f"Customers first seen in {WINDOW_START[:7]} are left-censored (history before the window is unknown)")

    status = q("""SELECT mth, lifecycle_status, count(*) n FROM cm WHERE NOT pre_signup GROUP BY ALL""")
    save(status.pivot(index="mth", columns="lifecycle_status", values="n").fillna(0).astype(int).reset_index(),
         "s6_lifecycle_status_by_month", show=12)

    lt3 = per_cust[(per_cust.n_active_months > 0) & (per_cust.n_active_months < MIN_HISTORY_MONTHS)].n_customers.sum()
    act = per_cust[per_cust.n_active_months > 0].n_customers.sum()
    flag("S6", f"{100 * lt3 / max(act, 1):.1f}% of active customers have < {MIN_HISTORY_MONTHS} active months "
               "-> weak trend history")

    save(q("""
        WITH x AS (SELECT customer_id, mth, gross_spend, n_eng, n_int,
                          lag(gross_spend) OVER w p_spend, lag(n_eng) OVER w p_eng
                   FROM cm WINDOW w AS (PARTITION BY customer_id ORDER BY mth)),
        y AS (SELECT 'gross_spend' metric, (coalesce(gross_spend,0) - p_spend) / p_spend chg FROM x WHERE p_spend > 0
              UNION ALL
              SELECT 'engagement_events', (coalesce(n_eng,0) - p_eng) / p_eng FROM x WHERE p_eng > 0)
        SELECT metric, count(*) n_pairs,
               round(100.0*count_if(chg <= -1)/count(*), 2) pct_dropped_to_zero,
               round(100.0*count_if(chg > -1 AND chg <= -0.3)/count(*), 2) pct_decline_gt_30,
               round(100.0*count_if(chg > -0.3 AND chg < 0.3)/count(*), 2) pct_stable,
               round(100.0*count_if(chg >= 0.3)/count(*), 2) pct_increase_gt_30,
               round(median(chg), 3) median_change
        FROM y GROUP BY 1"""), "s6_mom_change_buckets")

    save(q("""
        SELECT count(DISTINCT customer_id) FILTER (WHERE pre_signup AND active) customers_active_before_signup,
               count(*) FILTER (WHERE pre_signup AND active) customer_months_active_before_signup
        FROM cm"""), "s6_activity_before_signup")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(per_cust.n_active_months.astype(str), per_cust.n_customers)
    ax.set_xlabel("Active months in period"); ax.set_ylabel("Customers"); ax.set_title("History depth per customer")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s6_active_months_hist.png", dpi=120); plt.show()


run_step(step6_lifecycle)

## Step 7 – Interactions and sentiment (text, CSAT/NPS, labels)

In [ ]:
def step7_interactions():
    header("STEP 7 - Interactions, text, CSAT/NPS, sentiment labels")
    save(q("""SELECT channel, interaction_type, count(*) n FROM itx GROUP BY ALL ORDER BY n DESC"""),
         "s7_channel_by_type", show=0)
    save(q("""SELECT interaction_type, count(*) n,
                     round(100.0*count_if(sentiment_label='positive')/count(*), 2) pct_positive,
                     round(100.0*count_if(sentiment_label='neutral')/count(*), 2)  pct_neutral,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 2) pct_negative,
                     round(avg(sscore), 3) avg_sentiment_gt
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s7_sentiment_by_type")
    save(q("""SELECT channel, count(*) n,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 2) pct_negative,
                     round(100.0*count(csat)/count(*), 2) csat_fill_pct,
                     round(100.0*count(nps)/count(*), 2) nps_fill_pct,
                     round(avg(sscore), 3) avg_sentiment_gt
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s7_by_channel")
    save(q("""SELECT sentiment_label, count(*) n, round(min(sscore),3) min, round(quantile_cont(sscore,0.05),3) p05,
                     round(median(sscore),3) p50, round(quantile_cont(sscore,0.95),3) p95, round(max(sscore),3) max
              FROM itx GROUP BY 1 ORDER BY 1"""), "s7_label_vs_score")

    txt = save(q("""
        SELECT count(*) n, round(100.0*count_if(txt IS NULL OR trim(txt)='')/count(*), 2) blank_pct,
               round(avg(length(txt)), 1) avg_chars, quantile_cont(length(txt), 0.5) p50_chars,
               quantile_cont(length(txt), 0.95) p95_chars,
               round(100.0*count_if(len(string_split(trim(txt), ' ')) < 3)/count(*), 2) lt3_words_pct,
               round(100.0*count_if(regexp_matches(txt, '[^\\x00-\\x7F]'))/count(*), 2) non_ascii_pct,
               count(DISTINCT txt) n_distinct_texts,
               round(100.0*count(DISTINCT txt)/count(*), 2) distinct_text_pct
        FROM itx"""), "s7_text_quality")
    if txt.distinct_text_pct.iloc[0] < 20:
        flag("S7", f"Only {txt.distinct_text_pct.iloc[0]:.1f}% of texts are unique -> template-like text; "
                   "check for overfitting and memorisation in the sentiment model")
    top = save(q("""SELECT txt, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (), 3) pct
                    FROM itx GROUP BY 1 ORDER BY n DESC LIMIT 100"""), "s7_top_100_texts", show=5)
    flag("S7", f"Top 100 texts cover {top.pct.sum():.1f}% of interactions")

    save(q("""SELECT trim(split_part(txt, ',', 1)) first_clause, count(*) n,
                     round(100.0*count_if(sentiment_label='positive')/count(*), 1) pct_pos,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 1) pct_neg
              FROM itx GROUP BY 1 ORDER BY n DESC LIMIT 50"""), "s7_first_clause_label_mix", show=10)
    save(q("""SELECT trim(split_part(txt, ',', 2)) second_clause, count(*) n,
                     round(100.0*count_if(sentiment_label='positive')/count(*), 1) pct_pos,
                     round(100.0*count_if(sentiment_label='negative')/count(*), 1) pct_neg
              FROM itx GROUP BY 1 ORDER BY n DESC LIMIT 50"""), "s7_second_clause_label_mix", show=0)

    save(q("""SELECT round(100.0*count(csat)/count(*), 2) csat_fill_pct, round(100.0*count(nps)/count(*), 2) nps_fill_pct,
                     round(100.0*count_if(csat IS NOT NULL OR nps IS NOT NULL)/count(*), 2) any_survey_pct,
                     round(corr(sscore, csat), 3) corr_sentiment_csat, round(corr(sscore, nps), 3) corr_sentiment_nps,
                     round(corr(csat, nps), 3) corr_csat_nps
              FROM itx"""), "s7_survey_fill_and_correlation")
    save(q("""SELECT 'csat' metric, CAST(csat AS INT) score, count(*) n FROM itx WHERE csat IS NOT NULL GROUP BY ALL
              UNION ALL
              SELECT 'nps', CAST(nps AS INT), count(*) FROM itx WHERE nps IS NOT NULL GROUP BY ALL
              ORDER BY 1, 2"""), "s7_survey_distributions", show=0)
    save(q("""SELECT sentiment_label, round(avg(csat), 3) avg_csat, round(avg(nps), 3) avg_nps,
                     count(csat) n_csat, count(nps) n_nps
              FROM itx GROUP BY 1 ORDER BY 1"""), "s7_survey_by_sentiment_label")
    save(q("""SELECT resolution_status, count(*) n, round(avg(sscore), 3) avg_sentiment_gt,
                     round(avg(csat), 3) avg_csat, round(100.0*count_if(fcr)/count(*), 2) fcr_pct
              FROM itx GROUP BY 1 ORDER BY n DESC"""), "s7_resolution_vs_sentiment")

    # Stratified sample for manual reading (fill 'my_label' in the CSV)
    sample = q(f"""SELECT interaction_id, channel, interaction_type, resolution_status, txt AS interaction_text,
                          sentiment_label, sscore AS sentiment_score_gt, csat, nps, '' AS my_label, '' AS notes
                   FROM itx QUALIFY row_number() OVER (PARTITION BY sentiment_label
                                                       ORDER BY hash(interaction_id || '{SEED}')) <= {TEXT_SAMPLE_PER_LABEL}""")
    save(sample, "s7_manual_review_sample", show=5)

    lens = q("SELECT length(txt) l FROM itx USING SAMPLE 200000 ROWS")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(lens.l.dropna(), bins=50); ax.set_title("Interaction text length (chars, 200k sample)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "s7_text_length_hist.png", dpi=120); plt.show()


run_step(step7_interactions)

## Step 8 – Relationships: correlations and leading indicators

In [ ]:
FEATURES = ["n_tx", "n_tx_failed", "n_refunds", "spend_pctile", "n_eng", "n_logins", "eng_seconds",
            "pages_viewed", "n_conversions", "n_int", "n_complaint", "n_negative", "n_open_issue",
            "avg_sentiment_gt", "avg_csat", "avg_nps"]
NO_FILL = ("spend_pctile", "avg_sentiment_gt", "avg_csat", "avg_nps")   # averages: keep NULL, don't fill with 0


def step8_relationships():
    header("STEP 8 - Correlations and leading indicators (proxy labels)")
    cols = ", ".join(c if c in NO_FILL else f"coalesce({c}, 0) AS {c}" for c in FEATURES)
    df = q(f"""SELECT {cols} FROM (SELECT * FROM cm WHERE active)
               USING SAMPLE reservoir({CORR_SAMPLE_ROWS} ROWS) REPEATABLE ({SEED})""")
    corr = df.corr(method="spearman").round(3)
    save(corr.reset_index().rename(columns={"index": "feature"}), "s8_spearman_correlation", show=0)
    display(corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format("{:.2f}"))
    pairs = []
    for i, a in enumerate(FEATURES):
        for b in FEATURES[i + 1:]:
            r = corr.loc[a, b]
            if pd.notna(r) and abs(r) >= HIGH_CORR:
                pairs.append({"feature_a": a, "feature_b": b, "spearman": r})
                flag("S8", f"{a} ~ {b} (rho={r}) -> redundant; keep one")
    save(pd.DataFrame(pairs, columns=["feature_a", "feature_b", "spearman"]), "s8_high_corr_pairs")

    # Do complaints / negative / cancellation requests precede inactivity or decline?
    lead = save(q(f"""
        WITH x AS (
            SELECT customer_id, mth, active, n_int, n_complaint, n_negative, n_cancel_req,
                   coalesce(gross_spend, 0) spend, coalesce(n_eng, 0) eng,
                   lead(CAST(active AS INT), 1) OVER w active_next1,
                   max(CAST(active AS INT)) OVER (PARTITION BY customer_id ORDER BY mth
                                                  ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING) active_next3,
                   coalesce(lead(gross_spend, 1) OVER w, 0) spend_next1,
                   coalesce(lead(n_eng, 1) OVER w, 0) eng_next1
            FROM cm WINDOW w AS (PARTITION BY customer_id ORDER BY mth))
        SELECT CASE WHEN n_cancel_req > 0 THEN '1_cancellation_request'
                    WHEN n_complaint > 0 THEN '2_complaint_or_dispute'
                    WHEN n_negative > 0 THEN '3_negative_interaction'
                    WHEN coalesce(n_int, 0) > 0 THEN '4_other_interaction'
                    ELSE '5_no_interaction' END signal_in_month_M,
               count(*) n_customer_months,
               round(100.0*avg(active_next1), 2) pct_active_M1,
               round(100.0*(1 - avg(active_next3)), 2) pct_inactive_next_3_months,
               round(100.0*count_if(spend > 0 AND spend_next1 < 0.7*spend)/nullif(count_if(spend > 0), 0), 2)
                   pct_spend_drop_gt30_M1,
               round(100.0*count_if(eng > 0 AND eng_next1 < 0.7*eng)/nullif(count_if(eng > 0), 0), 2)
                   pct_eng_drop_gt30_M1
        FROM x
        WHERE active AND mth <= DATE '{WINDOW_END}' - INTERVAL 4 MONTH
        GROUP BY 1 ORDER BY 1"""), "s8_signal_vs_next_months")
    base = lead[lead.signal_in_month_M == "5_no_interaction"].pct_inactive_next_3_months
    canc = lead[lead.signal_in_month_M == "1_cancellation_request"].pct_inactive_next_3_months
    if len(base) and len(canc):
        lift = canc.iloc[0] / max(base.iloc[0], 1e-9)
        flag("S8", f"Cancellation request -> inactive next 3 months: {canc.iloc[0]:.1f}% vs {base.iloc[0]:.1f}% "
                   f"baseline (lift {lift:.2f}x). {'Usable' if lift >= 1.3 else 'Weak'} as churn proxy label")


run_step(step8_relationships)

## Step 9 – Segment behaviour (context only, never a penalty)

In [ ]:
SEGMENT_DIMS = ["customer_segment", "loyalty_tier", "region", "preferred_channel", "tenure_band", "age_band"]


def segment_view():
    con.execute("""
        CREATE OR REPLACE TEMP VIEW cm_seg AS
        SELECT m.*, c.customer_segment, c.loyalty_tier, c.region, c.preferred_channel,
               CASE WHEN c.tenure_n < 12 THEN '0-11m' WHEN c.tenure_n < 24 THEN '12-23m'
                    WHEN c.tenure_n < 48 THEN '24-47m' WHEN c.tenure_n IS NULL THEN 'unknown' ELSE '48m+' END tenure_band,
               CASE WHEN c.age_n < 25 THEN '<25' WHEN c.age_n < 35 THEN '25-34' WHEN c.age_n < 50 THEN '35-49'
                    WHEN c.age_n < 65 THEN '50-64' WHEN c.age_n IS NULL THEN 'unknown' ELSE '65+' END age_band
        FROM cm m JOIN cust1 c USING (customer_id)""")


def step9_segments():
    header("STEP 9 - Segment behaviour (decide global vs segment-relative scoring)")
    segment_view()
    metrics = {
        "pct_active": "100.0*avg(CAST(active AS INT))",
        "avg_tx_when_active": "avg(n_tx) FILTER (WHERE active)",
        "avg_spend_pctile": "avg(spend_pctile)",
        "avg_eng_when_active": "avg(coalesce(n_eng,0)) FILTER (WHERE active)",
        "pct_with_interaction": "100.0*count_if(has_int)/nullif(count_if(active),0)",
        "complaint_rate": "100.0*count_if(n_complaint > 0)/nullif(count_if(has_int),0)",
        "negative_rate": "100.0*count_if(n_negative > 0)/nullif(count_if(has_int),0)",
        "avg_sentiment_gt": "avg(avg_sentiment_gt)",
        "pct_2plus_sources": "100.0*count_if(n_sources >= 2)/nullif(count_if(active),0)",
    }
    sel = ", ".join(f"round({v}, 3) AS {k}" for k, v in metrics.items())
    frames, spread = [], []
    for dim in SEGMENT_DIMS:
        d = q(f"""SELECT '{dim}' dimension, CAST({dim} AS VARCHAR) segment,
                         count(DISTINCT customer_id) n_customers, {sel}
                  FROM cm_seg WHERE NOT pre_signup GROUP BY ALL ORDER BY n_customers DESC""")
        frames.append(d)
        for m in metrics:
            vals = d[m].dropna()
            vals = vals[vals > 0]
            if len(vals) > 1:
                ratio = vals.max() / vals.min()
                spread.append({"dimension": dim, "metric": m, "max_min_ratio": round(ratio, 3)})
                if ratio > SEGMENT_SPREAD_ALERT and m != "avg_sentiment_gt":
                    flag("S9", f"{m} differs by {ratio:.2f}x across {dim} -> score vs segment/own history")
    save(pd.concat(frames), "s9_segment_profile", show=20)
    save(pd.DataFrame(spread).sort_values("max_min_ratio", ascending=False), "s9_segment_spread", show=15)


run_step(step9_segments)

## Step 10 – Leakage register and eligibility funnel

In [ ]:
COLUMN_ROLES = {   # column -> (role, note); everything else defaults to 'feature candidate'
    "customer_id": ("id", "join key"), "transaction_id": ("id", ""), "engagement_id": ("id", ""),
    "interaction_id": ("id", ""), "agent_id": ("id", "not a customer feature"),
    "sentiment_label": ("label / evaluation only", "ground truth - never a feature"),
    "sentiment_score_gt": ("label / evaluation only", "ground truth - never a feature"),
    "csat_score": ("validation label", "use to validate sentiment; sparse"),
    "nps_score": ("validation label", "use to validate sentiment; sparse"),
    "tenure_months": ("context (snapshot)", "likely as-of a single date - recompute per month from signup_date"),
    "loyalty_tier": ("context (snapshot)", "may reflect end-of-period status - context only, never a penalty"),
    "age": ("context only", "do not lower health/sentiment"), "gender": ("context only", "do not lower health/sentiment"),
    "region": ("context / segmentation", ""), "country": ("context / segmentation", ""),
    "customer_segment": ("context / segmentation", ""), "income_band": ("context only", "do not lower health/sentiment"),
    "acquisition_channel": ("context", ""), "preferred_channel": ("context / action routing", ""),
    "marketing_consent": ("action constraint", "check before recommending outreach"),
    "month": ("time key", "re-derive from timestamp"), "signup_date": ("context", "used to derive tenure per month"),
}


def step10_leakage_eligibility():
    header("STEP 10 - Leakage register and eligibility funnel")
    rows = []
    for t in TABLES:
        for c in columns_of(f"raw_{t}"):
            role, note = COLUMN_ROLES.get(c, ("feature candidate", "aggregate to customer-month using data <= month M"))
            rows.append({"table": t, "column": c, "role": role, "note": note})
    save(pd.DataFrame(rows), "s10_column_role_register", show=0)
    flag("S10", "sentiment_label / sentiment_score_gt are ground truth -> evaluation only (label leakage otherwise)")

    ref = save(q("""SELECT CAST(date_trunc('month', signup + to_months(CAST(tenure_n AS INT))) AS DATE) implied_as_of_month,
                           count(*) n FROM cust WHERE signup IS NOT NULL AND tenure_n IS NOT NULL
                    GROUP BY 1 ORDER BY n DESC LIMIT 10"""), "s10_tenure_implied_as_of_date")
    if len(ref):
        top = pd.Timestamp(ref.implied_as_of_month.iloc[0])
        share = 100 * ref.n.iloc[0] / q("SELECT count(*) FROM cust WHERE signup IS NOT NULL").iloc[0, 0]
        flag("S10", f"tenure_months mostly implies as-of {top:%Y-%m} ({share:.0f}% of customers) -> "
                    "snapshot field; recompute tenure for each scoring month")

    save(q("""SELECT 'transactions' src, count_if(mth <> CAST(date_trunc('month', ts) AS DATE)) misassigned FROM tx
              UNION ALL SELECT 'engagements', count_if(mth <> CAST(date_trunc('month', ts) AS DATE)) FROM eng
              UNION ALL SELECT 'interactions', count_if(mth <> CAST(date_trunc('month', ts) AS DATE)) FROM itx"""),
         "s10_month_assignment_check", show=0)

    funnel = save(q(f"""
        SELECT mth,
               count(*) all_customers,
               count_if(NOT pre_signup) signed_up,
               count_if(active) active_in_month,
               count_if({HISTORY_COND}) active_with_{MIN_HISTORY_MONTHS}m_history,
               count_if({HISTORY_COND} AND n_sources >= 2) plus_2plus_sources,
               count_if({CORE_COND}) core_tx_plus_1,
               count_if({HISTORY_COND} AND n_sources = 1) low_confidence_1_source,
               count_if(active AND months_observed < {MIN_HISTORY_MONTHS}) new_or_short_history,
               count_if(lifecycle_status = 'dormant') dormant,
               count_if(lifecycle_status = 'churn_proxy') churn_proxy
        FROM cm GROUP BY 1 ORDER BY 1"""), "s10_eligibility_funnel_by_month", show=12)
    first_ok = funnel[funnel[f"active_with_{MIN_HISTORY_MONTHS}m_history"] > 0].mth.min()
    flag("S10", f"First month with scorable history: {pd.Timestamp(first_ok):%Y-%m} "
                f"(first {MIN_HISTORY_MONTHS - 1} months are warm-up only)")


run_step(step10_leakage_eligibility)

## Step 11 – Cohort selection scorecard
Weights below are starting assumptions – adjust `WEIGHTS` to match business priorities and re-run this cell.

In [ ]:
COHORT_DIMS = ["customer_segment", "loyalty_tier", "region", "preferred_channel", "tenure_band"]
WEIGHTS = {"value": 0.25, "coverage": 0.20, "labels": 0.15, "sample": 0.15,
           "history": 0.10, "stability": 0.10, "homogeneity": 0.05}


def step11_cohort():
    header("STEP 11 - Cohort selection scorecard")
    segment_view()
    frames = []
    dims = [("ALL", "'ALL'")] + [(d, f"CAST({d} AS VARCHAR)") for d in COHORT_DIMS]
    for dim, expr in dims:
        d = q(f"""
            SELECT '{dim}' dimension, {expr} segment,
                   count(DISTINCT customer_id) FILTER (WHERE {CORE_COND}) n_core_customers,
                   count_if({CORE_COND}) n_core_customer_months,
                   round(100.0*count_if({CORE_COND})/nullif(count_if({HISTORY_COND}), 0), 2) coverage_pct,
                   round(avg(active_months_to_date) FILTER (WHERE {CORE_COND}), 2) avg_active_months_history,
                   round(100.0*count_if({CORE_COND} AND (n_csat > 0 OR n_nps > 0))/nullif(count_if({CORE_COND}),0), 2) survey_label_pct,
                   round(100.0*count_if({CORE_COND} AND has_int)/nullif(count_if({CORE_COND}),0), 2) interaction_label_pct,
                   round(avg(spend_pctile) FILTER (WHERE {CORE_COND}), 4) avg_spend_pctile,
                   round(stddev(spend_pctile) FILTER (WHERE {CORE_COND}), 4) std_spend_pctile,
                   round(100.0*count_if({CORE_COND} AND (n_complaint > 0 OR n_negative > 0))/nullif(count_if({CORE_COND}),0), 2) risk_signal_pct
            FROM cm_seg GROUP BY ALL""")
        monthly = q(f"""SELECT {expr} segment, mth, count_if({CORE_COND}) n FROM cm_seg
                        WHERE months_observed >= {MIN_HISTORY_MONTHS} GROUP BY ALL ORDER BY 2""")
        monthly = monthly[monthly.n > 0]
        stab = (monthly.sort_values("mth").groupby("segment").n
                .apply(lambda s: (s.pct_change().abs() * 100).max()).rename("max_mom_swing_pct").reset_index())
        frames.append(d.merge(stab, on="segment", how="left"))
    sc = pd.concat(frames, ignore_index=True)

    cand = sc[sc.dimension != "ALL"].copy()
    cand["passes_sample_gate"] = cand.n_core_customers >= MIN_COHORT_CUSTOMERS
    rank = lambda s, asc=True: s.rank(pct=True, ascending=asc)
    # value = spend level (within-currency percentile) + how much risk there is to act on
    cand["s_value"] = rank(cand.avg_spend_pctile) * 0.5 + rank(cand.risk_signal_pct) * 0.5
    cand["s_coverage"] = rank(cand.coverage_pct)
    cand["s_labels"] = rank(cand.survey_label_pct) * 0.5 + rank(cand.interaction_label_pct) * 0.5
    cand["s_sample"] = rank(cand.n_core_customers)
    cand["s_history"] = rank(cand.avg_active_months_history)
    cand["s_stability"] = rank(cand.max_mom_swing_pct, asc=False)
    cand["s_homogeneity"] = rank(cand.std_spend_pctile, asc=False)
    cand["cohort_score"] = sum(cand[f"s_{k}"] * w for k, w in WEIGHTS.items()).round(3)
    cand.loc[~cand.passes_sample_gate, "cohort_score"] = 0
    cand = cand.sort_values("cohort_score", ascending=False)

    save(sc[sc.dimension == "ALL"], "s11_core_cohort_overall")
    save(cand, "s11_cohort_scorecard", show=15)
    top = cand.iloc[0]
    flag("S11", f"Top candidate cohort: {top.dimension} = {top.segment} (score {top.cohort_score}, "
                f"{top.n_core_customers:,} core customers, coverage {top.coverage_pct}%). "
                "Confirm with business value before final selection.")


run_step(step11_cohort)

## Summary – all auto-flagged findings

In [ ]:
findings = pd.DataFrame(FINDINGS, columns=["step", "finding"]).drop_duplicates()
with pd.option_context("display.max_colwidth", None):
    display(findings)

lines = ["# EDA summary - Customer Health & Sentiment Scoring", "",
         f"Data window: {WINDOW_START} to {WINDOW_END} (exclusive) | grain: customer_id x month", "",
         "## Auto-flagged findings", ""]
lines += [f"- **{s}** {m}" for s, m in findings.itertuples(index=False)]
lines += ["", "## Output files", ""] + [f"- {p.name}" for p in sorted(OUT_DIR.glob("*.csv"))]
(OUT_DIR / "eda_summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Summary written to {OUT_DIR / 'eda_summary.md'} ({len(findings)} findings)")

In [ ]:
con.close()   # release the DuckDB file lock when finished